# Polynomial-cost orbital optimization of cluster number quasisymetries

In [31]:
import numpy as np

# Molecule input parameters
molecule = 'h2o' # 'n2'      # supports: h2, h2o, n2, lih, h4_linear, h4_square
bond_length = .96 # 1.1     # Angstrom
basis = 'sto3g' # '6-31g'

# DMRG parameters
bond_dim = 50 # 500
n_sweeps = 10 # 20

# High-level input parameters
initial_basis = 'MOs' # 'MOs': HF molecular orbitals, #TODO: 'NatOs': natural orbitals, 'LocOs': orbitals localized on nuclei via...
use_custom_cluster_matrix = True # True: set your cluster matrix below, #TODO False: automatic cluster_matrix optimization, set num_clusters below
num_transfers = 2 # maximal number of inter-cluster electron transfers starting from fundamental sector
type_cost_function = 'variance' # 'variance' or #TODO: 'eval_eq'
if use_custom_cluster_matrix:
    cluster_matrix = np.array([ # num_clusters x norb binary matrix with custom clusters. Clusters should not overlap nor use all orbitals.
    [1, 1, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 0]
])
    # Calculate the sum of each column (one column per orbital)
    column_sums = np.sum(cluster_matrix, axis=0)
    if not np.all(np.isin(column_sums, [0, 1])):
        invalid_columns = np.where((column_sums != 0) & (column_sums != 1))[0]
        raise ValueError(f"Error: Orbitals {invalid_columns} should appear in at most one cluster.")
    if np.all(column_sums == 1):
        raise ValueError(f"Error: The clusters cover all orbitals. Remove one cluster to avoid redundancy.")
    num_clusters = len(cluster_matrix)
else:
    num_clusters = 3 # specify number of clusters

In [32]:
# import new source code

import sys
sys.path.insert(0, '..')
import src.cluster_number_operators as cluster_ops

In [33]:
# ============================================================================
# SECTION 1: Get psi_mps with DMRG using MOs as sites
# ============================================================================
import pyscf
from chemistry import get_geometry_and_description
from src.dmrg_solver import Block2DMRGSolver, DMRGConfig, solve_or_load_ground_state
import tempfile

# --- Step 1.1: Build molecule and run HF ---
geometry, _ = get_geometry_and_description(molecule, bond_length)
mol = pyscf.M(atom=geometry, basis=basis)
mf = pyscf.scf.RHF(mol)
mf.kernel()

norb, nelec = mol.nao, mol.nelec
print(f"Number of orbitals: {norb}")
h1e = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
g2e = pyscf.ao2mo.full(mol, mf.mo_coeff) # compressed
# recall: chemist notation (pq|rs) = int d1 d2 \bar phi_p(1) phi_q(1) 1/r_12 \bar phi_r(2) phi_s(2) = <p, r| W |q, s>
ecore = mol.energy_nuc()

# --- Step 1.2: Run DMRG ---
store_dir = tempfile.mkdtemp(prefix='dmrg_')
solver = Block2DMRGSolver( # handles nelec as integer or pair
    h1e=h1e, g2e=g2e, ecore=ecore,
    n_elec=nelec, spin=mol.spin,
    store_dir=store_dir, n_threads=1
)
result = solve_or_load_ground_state(
    solver,
    config=DMRGConfig(max_bond_dim=bond_dim, n_sweeps=n_sweeps),
    reuse=False
)
dmrg_energy = result.energy
print(f"DMRG Energy: {dmrg_energy} Ha")

converged SCF energy = -74.9633190525401
Number of orbitals: 7
DMRG Energy: -75.01315470148963 Ha


In [34]:
# ============================================================================
# SECTION 2: Get 1- and 2-RDMs from the mps
# ============================================================================

# --- Step 2.1: Extract RDMs directly from MPS ---

# get mps
mps_tag = result.mps_tag
available_tags = solver.stored_tags()
if mps_tag not in available_tags:
    raise RuntimeError(
        f"Expected MPS tag '{mps_tag}' not found in store "
        f"{solver.store_dir}; available tags: {available_tags}"
    )
mps = solver.get_mps(tag=mps_tag)

# get rdms
rdm1_a, rdm1_b = solver.driver.get_1pdm(mps)
rdm2_aa, rdm2_ab, rdm2_bb = rdm2[0], rdm2[1], rdm2[2] = solver.driver.get_2pdm(mps)
# Formulas:
#
# rdm1_sigma[p, q] = <mps|a^dag_{p sigma}a_{q sigma}|mps>
#
# rdm2_aa[p, q, r, s] = <mps| a^dag_{p,alpha} a^dag_{q,alpha} a_{r,alpha} a_{s,alpha} |mps>   (alpha-alpha)
# rdm2_ab[p, q, r, s] = <mps| a^dag_{p,alpha} a^dag_{q,beta}  a_{r,beta}  a_{s,alpha} |mps>   (alpha-beta)
# rdm2_bb[p, q, r, s] = <mps| a^dag_{p,beta}  a^dag_{q,beta}  a_{r,beta}  a_{s,beta}  |mps>   (beta-beta)
#
# if needed, rdm2_ba[p, q, r, s] = <mps| a^dag_{p,beta} a^dag_{q,alpha}  a_{r,alpha}  a_{s,beta} |mps> (beta-alpha)
# is for any state given by:
# rdm2_ba[p, q, r, s] = rdm2_ab[q, p, s, r]

print(f"rdm1, type: {type(rdm1_alpha)}, shape: {np.shape(rdm1_alpha)}, trace: {np.trace(rdm1_alpha).round(8)}")
# print("rdm1_alpha:\n", rdm1_alpha.round(2))
print(f"Length of rdm2: {len(rdm2)}, shape of rdm2_aa: {np.shape(rdm2_aa)}")

# --- Step 2.2: Electronic energy reconstruction (sanity check) ---

#   E_1 = sum_pq h1e_pq (rdm1_a_pq + rdm1_b_pq)
#
#   E_2 = 1/2 sum_pqrs g2e_full[p,s,q,r] * rdm2_aa[p,q,r,s]
#       +     sum_pqrs g2e_full[p,s,q,r] * rdm2_ab[p,q,r,s]
#       + 1/2 sum_pqrs g2e_full[p,s,q,r] * rdm2_bb[p,q,r,s]
#   last line accounts for ab and ba; reason is rdm2_ba[p, q, r, s] = rdm2_ab[q, p, s, r]
#   and g2e_full[p, q, r, s] = g2e_full[r, s, p, q] for particle exchange symmetry of coulomb =
#   (...also = 6 more mat el.s for real orbitals! "8-fold permutational symmetry").
#   And these two things are independent of orbitals being real! (Worth a double check if ever needed.)
#
#   E_elec = E_1 + E_2
#   E_total = E_elec + ecore

g2e_full = pyscf.ao2mo.restore(1, g2e, norb) # not compressed; chemist's notatio

# One-body: direct contraction, no index permutation needed
e1 = np.einsum('pq,pq->', h1e, rdm1_a) + np.einsum('pq,pq->', h1e, rdm1_b)

# Two-body: 'psqr,pqrs->' index permutation needed - human checked!
e2 = (
    0.5 * np.einsum('psqr,pqrs->', g2e_full, rdm2_aa)      # alpha-alpha
    + np.einsum('psqr,pqrs->', g2e_full, rdm2_ab)          # alpha-beta + beta-alpha
    + 0.5 * np.einsum('psqr,pqrs->', g2e_full, rdm2_bb)    # beta-beta
)

e_check = e1 + e2 + ecore

print(f"Energy from RDMs - energy from DMRG: {(e_check - result.energy):.2e} Ha")

rdm1, type: <class 'numpy.ndarray'>, shape: (7, 7), trace: 5.0
Length of rdm2: 3, shape of rdm2_aa: (7, 7, 7, 7)
Energy from RDMs - energy from DMRG: 4.26e-14 Ha


In [35]:
# ============================================================================
# SECTION 2: Get best clusters, described via a binary cluster_matrix
# (if custom_cluster_matrix = None). Should use adaptation of Praveen's beam search.
# Can also use a Fiedler-like preliminary ordering of the orbitals, before beam search.
# See Praveen's thesis -> 4.5 Indexing symmetries with Fiedler reordering) 
# Careful: correlations of two orbitals in two different clusters
# are not a problem (e.g. H2); you really want small cluster number fluctuations...
# Still, some cheap two-orbital information-based preprocessing can help.
# ============================================================================
if use_custom_cluster_matrix == False:
    #TODO
    cluster_matrix = []

In [36]:
# ============================================================================
# SECTION 3: Get the optimal orbital basis, starting from initial_basis.
# Optimization cost function type is given by type_cost_function.
# ============================================================================

D = rdm1_a + rdm1_b
Gamma = rdm2_aa + rdm2_bb + rdm2_ab + rdm2_ab.transpose(1, 0, 3, 2)

if initial_basis == 'NatOs':
    #TODO
    D = []
    Gamma = []
if initial_basis == 'LocOs':
    #TODO
    D = []
    Gamma = []

# permute once: Gamma_tilde[k,n,l,m] = Gamma[k,l,m,n]
Gamma_tilde = Gamma.transpose(0, 3, 1, 2).reshape(norb * norb, norb * norb)

def get_cluster_indices(cluster_matrix, norb):
    """Convert the binary cluster_matrix into a list of orbital-index arrays,
    one per cluster, adding back the "ghost" cluster of uncovered orbitals.
    Precompute this once (it only depends on cluster_matrix, not on U)."""
    covered = np.any(cluster_matrix, axis=0)
    clusters = [np.where(row)[0] for row in cluster_matrix]
    ghost = np.where(~covered)[0]
    if ghost.size > 0:
        clusters.append(ghost)
    return clusters

def make_evaluator(D, Gamma_tilde, norb, cluster_matrix):
    # Precomputed once: list of orbital-index arrays, one per cluster
    # (including the ghost cluster of orbitals not covered by cluster_matrix).
    clusters = get_cluster_indices(cluster_matrix, norb)

    def evaluate(U):
        Uc = U.conj()

        # n1(U)_p = sum_kl U*[p,k] U[p,l] D[k,l]   -- unrestricted, needed for all p
        n1 = np.einsum('pk,pl,kl->p', Uc, U, D, optimize=True).real

        # Step 1: contract k -> new axis p  (matmul, O(norb^5)), all p needed
        #   T[p, n, l, m] = sum_k U*[p,k] Gamma_tilde[k,n,l,m]
        T = (Uc @ Gamma_tilde.reshape(norb, norb * norb * norb)).reshape(
            norb, norb, norb, norb
        )

        # Step 2: contract n, batched over p  (O(norb^4)), all p needed
        #   M[p, l, m] = sum_n T[p,n,l,m] U[p,n]
        M = np.einsum('pnlm,pn->plm', T, U, optimize=True)

        # Step 3 (restricted + symmetry-reduced): for each cluster, only
        # compute N[p,q] for p, q both in that cluster, and only for p <= q
        # (using N_pq = N_qp), then mirror. Zero outside clusters.
        N = np.zeros((norb, norb), dtype=M.dtype)
        for P, Q in cluster_pairs:
            if P.size == 0:
                continue
            # Gather only the rows/entries needed for these p<=q pairs.
            # M[P] reuses already-computed M (no recomputation), just indexing.
            M_pairs = M[P]     # (npairs, norb, norb): M[p,:,:] for each pair
            Uc_pairs = Uc[Q]   # (npairs, norb): U*[q,:] for each pair
            U_pairs = U[Q]     # (npairs, norb): U[q,:] for each pair

            # N_vals[n] = sum_lm M[p,l,m] * U*[q,l] * U[q,m]  for pair n=(p,q)
            N_vals = np.einsum('nlm,nl,nm->n', M_pairs, Uc_pairs, U_pairs,
                                optimize=True)

            N[P, Q] = N_vals
            N[Q, P] = N_vals.conj()  # N_qp = N_pq* in general;
            # for p == q this just overwrites the same (real) entry harmlessly

        n2 = N.real + np.diag(n1)
        return n1, n2

    return evaluate
    

In [37]:
# ============================================================================
# SECTION 4: Get labels of dominant sector and sectors obtained from it by 
# moving up to num_transfers electrons
# ============================================================================

In [38]:
# ============================================================================
# SECTION 5: For small systems, evaluate quality of sector decomposition
# by plotting K_sectors -> ΔE, K_states -> ΔE
# ============================================================================